# 9. Other algorithms

**The question:** base64 only moves bits. What does the builder do for algorithms that compute something?

It uses the cheapest of three forms for the MLP:

| Form | Used when | Example |
|---|---|---|
| Wiring | each input bit lands in one output bit | hex, base64 |
| Bit functions | each output bit depends on at most 4 input bits | XOR |
| Lookup table | otherwise, up to 4,096 units per output position | ROT13 |

In [1]:
import helpers
from helpers import load
import tasks

for family, task in tasks.every_task():
  print(f'{family:<11}{task.name:<11}', ', '.join(m.name for m in task.modes))

carry      add         add
carry      checksum    xor
carry      parity      parity
key        vigenere    encrypt, decrypt
key        xor         xor1, xor4
rearrange  base32      decode, encode
rearrange  base64      decode, encode
rearrange  base64url   decode, encode
rearrange  binary      decode, encode
rearrange  bits        reverse, nibbles
rearrange  byteswap    swap16, swap32
rearrange  hex         decode, encode
substitute atbash      atbash
substitute caesar      encrypt, decrypt
substitute case        upper, lower
substitute rot13       rot13


## 1. Wiring: hex

One byte in, two digits out. Each digit is 4 bits of the byte.

In [2]:
hexm = load('hex')
print(' '.join(hexm.generate('encode', b'Hi')))
for u in hexm.r.mlp.units:
  if u.startswith('encode'):
    print(u)

4 8 6 9
encode place 0: byte.bit4 -> out bit 0
encode place 0: byte.bit5 -> out bit 1
encode place 0: byte.bit6 -> out bit 2
encode place 0: byte.bit7 -> out bit 3
encode place 1: byte.bit0 -> out bit 0
encode place 1: byte.bit1 -> out bit 1
encode place 1: byte.bit2 -> out bit 2
encode place 1: byte.bit3 -> out bit 3
encode place 0: no more bytes -> END


## 2. Bit functions: XOR with a key

The output bit is `message bit XOR key bit`. No wire can do that, because the output depends on two bits. The builder
finds which bits each output depends on, by flipping them, and writes one unit per pattern that gives a 1.

In [3]:
xor = load('xor')
print(len(xor.r.mlp.units), 'units')
for u in [u for u in xor.r.mlp.units if u.startswith('xor1 place 0') and 'out bit 0' in u]:
  print(u)
print()
print(' '.join(xor.generate('xor1', b'Khello')), '  key K')

85 units
xor1 place 0: message.bit0=1 key.bit0=0 -> out bit 0
xor1 place 0: message.bit0=0 key.bit0=1 -> out bit 0

x23 x2e x27 x27 x24   key K


Two units per output bit: bit 0 of the output is 1 when the message and key bits differ.

## 3. Lookup table: ROT13

Each byte maps to another, with no pattern in the bits. So the MLP is a table: one unit per byte value.

In [4]:
rot = load('rot13')
print(len(rot.r.mlp.units), 'units')
for u in rot.r.mlp.units[65:68]:
  print(u)
print(''.join(chr(int(t[1:], 16)) for t in rot.generate('rot13', b'Hello Darpan!')))

257 units
rot13 place 0: byte x41 -> x4e
rot13 place 0: byte x42 -> x4f
rot13 place 0: byte x43 -> x50
Uryyb Qnecna!


## 4. State: addition

Addition needs a carry, and the model has no memory. It reads its own previous output instead: the carry into a digit can
be worked out from the previous digit's inputs and the sum digit the model wrote there.

In [5]:
add = load('add')
m = add.r.task.mode('add')
print('reads:', m.fetch, ' of which earlier outputs:', m.from_outputs)
print('1011 + 0110 =', ' '.join(add.generate('add', '1011+0110')), '  (least significant digit first: 10001 = 17)')

reads: ('a', 'b', 'a_prev', 'b_prev', 'sum_prev')  of which earlier outputs: ('sum_prev',)
1011 + 0110 = 1 0 0 0 1   (least significant digit first: 10001 = 17)


## 5. All 16

Every task builds from a file of a few lines. Every one is proved by `python tests/tasks.py --prove`.

In [6]:
for family, task in tasks.every_task():
  r = load(task.name).r
  print(f'{task.name:<11}{len(r.heads):>3} heads {len(r.mlp.units):>5} units')

add          6 heads    33 units
checksum     3 heads    17 units
parity       3 heads     3 units
vigenere     5 heads  5416 units
xor          5 heads    85 units
base32       6 heads    92 units
base64       5 heads    54 units
base64url    5 heads    54 units
binary      10 heads    18 units
bits         3 heads    18 units
byteswap     3 heads    50 units
hex          4 heads    18 units
atbash       2 heads   257 units
caesar       3 heads   514 units
case         3 heads   514 units
rot13        2 heads   257 units
